# Chapter 5 — NLP Layer & Loss Toolbox (Practice)

Work through these exercises **after reading** `notes/ch05-nlp-layer-and-loss-toolbox.md`.

Each exercise states the *decision you're practicing*, gives a stub cell to fill in, and is followed by a pre-written **verification cell** — run it to grade yourself. Hand-write your answers in your working copy under `solutions/` (this `template/` copy stays pristine), and don't peek at `solved/` until the verification passes or you're genuinely stuck.

In [1]:
# ============================================================
# TOPIC: The NLP layer & loss toolbox — Embedding, LayerNorm, Dropout, CE
# MATH:  CE = -log softmax(z)_t;  LayerNorm: (x - mean) / sqrt(var + eps) * gamma + beta
# REF:   B00 ch05 notes — nlp-layer-and-loss-toolbox
# ============================================================

# --- Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Reproducibility & device ---
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version : {torch.__version__}")
print(f"device        : {device}  (every exercise here runs fine on CPU)")

torch version : 2.13.0
device        : cpu  (every exercise here runs fine on CPU)


## Exercise 1 — Embedding = One-Hot @ Weight

Prove the notes §2 identity: an embedding lookup is exactly a one-hot vector times the weight matrix. Implement `lookup_via_onehot` with `F.one_hot` and `@`, and match `nn.Embedding`'s output to the bit.

**Decision you're practicing:** knowing what `nn.Embedding` *is* (learnable row selection) — which is why it needs `int64` inputs and trains like any Linear.

In [2]:
torch.manual_seed(0)
VOCAB, DIM = 6, 4
embedding = nn.Embedding(VOCAB, DIM)
token_ids = torch.tensor([[2, 0, 5],
                          [1, 1, 3]])          # shape: (batch=2, seq=3)

def lookup_via_onehot(embedding_layer, ids):
    """Reproduce embedding_layer(ids) using ONLY F.one_hot and a matmul."""
    one_hot = F.one_hot(ids, num_classes=embedding_layer.num_embeddings)   # (B, T, V) int64
    one_hot = one_hot.to(embedding_layer.weight.dtype)                     # matmul needs floats
    return one_hot @ embedding_layer.weight                                # (B,T,V) @ (V,D) → (B,T,D)

via_lookup = embedding(token_ids)
via_onehot = lookup_via_onehot(embedding, token_ids)
print(f"lookup  : {tuple(via_lookup.shape)}  # (batch, seq, dim)")
print(f"one-hot : {tuple(via_onehot.shape)}")
print(f"token 2's vector, both ways:\n  {via_lookup[0, 0].tolist()}\n  {via_onehot[0, 0].tolist()}")

lookup  : (2, 3, 4)  # (batch, seq, dim)
one-hot : (2, 3, 4)
token 2's vector, both ways:
  [0.46809640526771545, -0.1577124446630478, 1.4436601400375366, 0.26604941487312317]
  [0.46809640526771545, -0.1577124446630478, 1.4436601400375366, 0.26604941487312317]


**Verification**

In [3]:
# --- Verification: Exercise 1 ---
result = lookup_via_onehot(embedding, token_ids)
assert result is not None, "fill in the stub above first"
assert result.shape == (2, 3, 4), f"wrong shape {tuple(result.shape)}"
assert torch.allclose(result, embedding(token_ids), atol=1e-6), \
    "one-hot @ weight must equal the lookup exactly — same math, different cost"

# and the reason int64 is mandatory: a lookup can't fetch row 2.7
try:
    embedding(token_ids.float())
    raise AssertionError("embedding accepted float ids?!")
except (RuntimeError, TypeError) as err:
    print(f"float ids rejected, as expected: {str(err)[:80]}...")
print("Exercise 1 passed ✓")

float ids rejected, as expected: Expected tensor for argument #1 'indices' to have one of the following scalar ty...
Exercise 1 passed ✓


## Exercise 2 — What `padding_idx` Actually Guarantees

Build an embedding with `padding_idx=0`, run a forward + backward + optimizer step on a batch **containing pad tokens**, and capture four artifacts: the pad row before, its gradient, a real token's gradient, and the pad row after the step.

**Decision you're practicing:** `padding_idx` = zero row + **permanently zero gradient** — the pad vector can never drift, no matter how much it appears in batches.

In [4]:
PAD_ID = 0
torch.manual_seed(1)
pad_embedding = nn.Embedding(10, 4, padding_idx=PAD_ID)
batch_ids = torch.tensor([[3, 3, 5, PAD_ID, PAD_ID]])     # shape: (1, 5) — two pads

pad_row_before = pad_embedding.weight[PAD_ID].clone()
print(f"pad row at init          : {pad_row_before.tolist()}   ← initialized to zeros")

optimizer = torch.optim.SGD(pad_embedding.parameters(), lr=1.0)
loss = pad_embedding(batch_ids).sum()      # every used row gets gradient 1 per appearance
loss.backward()

grad_pad_row = pad_embedding.weight.grad[PAD_ID].clone()
grad_real_row = pad_embedding.weight.grad[3].clone()
print(f"grad of pad row          : {grad_pad_row.tolist()}   ← forced to zero")
print(f"grad of row 3 (used 2×)  : {grad_real_row.tolist()}   ← real rows accumulate normally")

optimizer.step()
pad_row_after_step = pad_embedding.weight[PAD_ID].clone()
print(f"pad row after SGD step   : {pad_row_after_step.tolist()}   ← still zeros, forever")

pad row at init          : [0.0, 0.0, 0.0, 0.0]   ← initialized to zeros


grad of pad row          : [0.0, 0.0, 0.0, 0.0]   ← forced to zero
grad of row 3 (used 2×)  : [2.0, 2.0, 2.0, 2.0]   ← real rows accumulate normally
pad row after SGD step   : [0.0, 0.0, 0.0, 0.0]   ← still zeros, forever


**Verification**

In [5]:
# --- Verification: Exercise 2 ---
for name, value in [("pad_row_before", pad_row_before), ("grad_pad_row", grad_pad_row),
                    ("grad_real_row", grad_real_row), ("pad_row_after_step", pad_row_after_step)]:
    assert value is not None, f"{name} not captured — fill in the stub"

assert torch.equal(pad_row_before, torch.zeros(4)), "padding_idx row must start as zeros"
assert torch.equal(grad_pad_row, torch.zeros(4)), "the pad row's gradient must be exactly zero"
assert torch.equal(grad_real_row, torch.full((4,), 2.0)), \
    "row 3 appears twice with a sum loss → its gradient should be exactly 2s"
assert torch.equal(pad_row_after_step, torch.zeros(4)), \
    "after an lr=1.0 step the pad row must STILL be zeros — that is the guarantee"
print("pad row: zero at init, zero gradient, zero after training ✓")
print("Exercise 2 passed ✓")

pad row: zero at init, zero gradient, zero after training ✓
Exercise 2 passed ✓


## Exercise 3 — LayerNorm from Scratch

Implement `ManualLayerNorm`: per-token mean and **biased** variance over the last dimension (`unbiased=False` — the gotcha), normalize with `eps` inside the square root, then apply learned `gamma`/`beta`. Match `nn.LayerNorm` to 6 decimals on a `(batch, seq, dim)` tensor.

**Decision you're practicing:** what LayerNorm actually computes, per token — plus ch01's `keepdim=True` choreography in real use.

In [6]:
class ManualLayerNorm(nn.Module):
    """LayerNorm over the last dimension: gamma * (x - mean) / sqrt(var + eps) + beta."""
    def __init__(self, normalized_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(normalized_dim))    # learned scale — starts as identity
        self.beta = nn.Parameter(torch.zeros(normalized_dim))    # learned shift — starts as no-op

    def forward(self, inputs):
        """inputs: (..., normalized_dim) — normalize each token independently."""
        mean = inputs.mean(dim=-1, keepdim=True)                             # (..., 1)
        variance = inputs.var(dim=-1, keepdim=True, unbiased=False)          # BIASED: divide by d, not d-1
        normalized = (inputs - mean) / torch.sqrt(variance + self.eps)       # broadcast over the last dim
        return self.gamma * normalized + self.beta

demo = ManualLayerNorm(3)
dry_run = demo(torch.tensor([1.0, 2.0, 3.0]))
print(f"notes §3 dry-run [1,2,3] → {[round(v, 4) for v in dry_run.tolist()]}   # expect [-1.2247, 0, 1.2247]")

notes §3 dry-run [1,2,3] → [-1.2247, 0.0, 1.2247]   # expect [-1.2247, 0, 1.2247]


**Verification**

In [7]:
# --- Verification: Exercise 3 ---
manual_norm = ManualLayerNorm(8)
reference_norm = nn.LayerNorm(8)

torch.manual_seed(2)
test_inputs = torch.randn(2, 3, 8)                       # (batch, seq, dim)
mine = manual_norm(test_inputs)
assert mine is not None, "fill in the stub above first"
official = reference_norm(test_inputs)
assert torch.allclose(mine, official, atol=1e-6), \
    "mismatch vs nn.LayerNorm — most common cause: unbiased variance (PyTorch uses BIASED here)"

# the notes dry-run, exactly
dry_run = ManualLayerNorm(3)(torch.tensor([1.0, 2.0, 3.0]))
assert torch.allclose(dry_run, torch.tensor([-1.2247, 0.0, 1.2247]), atol=1e-4)

# per-token independence: each output row has mean ~0, std ~1 regardless of neighbors
assert torch.allclose(mine.mean(dim=-1), torch.zeros(2, 3), atol=1e-5)
print("matches nn.LayerNorm, reproduces the hand dry-run, normalizes per token ✓")
print("Exercise 3 passed ✓")

matches nn.LayerNorm, reproduces the hand dry-run, normalizes per token ✓
Exercise 3 passed ✓


## Exercise 4 — The Batch-Dependence Experiment

The **same sample** goes through normalization twice — once batched with tame companions, once with wild ones (mean 10, std 5). Implement `normalize_in_batch`, run it for `nn.LayerNorm` and `nn.BatchNorm1d` (train mode), and record your `verdict`.

If a layer gives the same sample **different outputs depending on its batch-mates**, it is batch-dependent — and wrong for sequence models.

**Decision you're practicing:** the LayerNorm-vs-BatchNorm axis choice, demonstrated rather than memorized.

In [8]:
the_sample   = torch.tensor([[1.0, 2.0, 3.0, 4.0]])       # shape (1, 4) — the SAME sample, always
torch.manual_seed(3)
companions_a = torch.randn(3, 4)                           # tame batch-mates
companions_b = torch.randn(3, 4) * 5 + 10                  # wild batch-mates

def normalize_in_batch(norm_layer, sample, companions):
    """Concatenate sample + companions into one batch, normalize, return THE SAMPLE's row."""
    batch = torch.cat([sample, companions], dim=0)         # (1+3, 4) — sample is row 0
    return norm_layer(batch)[0]

layer_norm = nn.LayerNorm(4)
batch_norm = nn.BatchNorm1d(4).train()                     # train mode = normalize with BATCH stats

ln_with_a = normalize_in_batch(layer_norm, the_sample, companions_a)
ln_with_b = normalize_in_batch(layer_norm, the_sample, companions_b)
bn_with_a = normalize_in_batch(batch_norm, the_sample, companions_a)
bn_with_b = normalize_in_batch(batch_norm, the_sample, companions_b)

print(f"LayerNorm, tame batch : {[round(v, 4) for v in ln_with_a.tolist()]}")
print(f"LayerNorm, wild batch : {[round(v, 4) for v in ln_with_b.tolist()]}   ← identical: per-token universe")
print(f"BatchNorm, tame batch : {[round(v, 4) for v in bn_with_a.tolist()]}")
print(f"BatchNorm, wild batch : {[round(v, 4) for v in bn_with_b.tolist()]}   ← the SAME sample, different answer!")

verdict = {
    "layer_norm_batch_dependent": False,
    "batch_norm_batch_dependent": True,
}

LayerNorm, tame batch : [-1.3416, -0.4472, 0.4472, 1.3416]
LayerNorm, wild batch : [-1.3416, -0.4472, 0.4472, 1.3416]   ← identical: per-token universe
BatchNorm, tame batch : [0.9366, 1.2623, 1.7171, 1.5631]
BatchNorm, wild batch : [-1.2197, -1.2389, -1.7144, -1.6636]   ← the SAME sample, different answer!


**Verification**

In [9]:
# --- Verification: Exercise 4 ---
for name, value in [("ln_with_a", ln_with_a), ("ln_with_b", ln_with_b),
                    ("bn_with_a", bn_with_a), ("bn_with_b", bn_with_b)]:
    assert value is not None, f"{name} not computed — fill in the stub"

assert torch.allclose(ln_with_a, ln_with_b, atol=1e-6), \
    "LayerNorm must give the sample IDENTICAL output regardless of batch-mates"
assert not torch.allclose(bn_with_a, bn_with_b, atol=1e-3), \
    "BatchNorm should give DIFFERENT outputs — its statistics come from the batch"
assert verdict["layer_norm_batch_dependent"] is False
assert verdict["batch_norm_batch_dependent"] is True
print("demonstrated: LayerNorm is batch-independent, BatchNorm is not → sequences use LayerNorm ✓")
print("Exercise 4 passed ✓")

demonstrated: LayerNorm is batch-independent, BatchNorm is not → sequences use LayerNorm ✓
Exercise 4 passed ✓


## Exercise 5 — The Forgotten `eval()` Detective

`buggy_predict` below serves a model with dropout **without switching modes** — the pre-written cell shows the symptom: the same input, different outputs every call. Write `correct_predict` with **both** inference switches: `model.eval()` (behavior) and `torch.no_grad()` (recording).

**Decision you're practicing:** `eval()` vs `no_grad()` — two different switches, and inference needs both.

In [10]:
torch.manual_seed(4)
serving_model = nn.Sequential(nn.Linear(8, 8), nn.Dropout(p=0.5))
probe = torch.randn(4, 8)

def buggy_predict(model, inputs):
    return model(inputs)                      # ← forgot eval() AND no_grad()

first_call = buggy_predict(serving_model, probe)
second_call = buggy_predict(serving_model, probe)
print(f"same input, two buggy calls identical? {torch.equal(first_call, second_call)}   ← the symptom")

def correct_predict(model, inputs):
    """Deterministic, graph-free inference."""
    model.eval()                              # behavior switch: dropout becomes identity
    with torch.no_grad():                     # recording switch: no graph, no wasted memory
        return model(inputs)

clean_1 = correct_predict(serving_model, probe)
clean_2 = correct_predict(serving_model, probe)
print(f"two correct calls identical?           {torch.equal(clean_1, clean_2)}")
print(f"output carries a graph?                {clean_1.grad_fn is not None}")

same input, two buggy calls identical? False   ← the symptom
two correct calls identical?           True
output carries a graph?                False


**Verification**

In [11]:
# --- Verification: Exercise 5 ---
serving_model.train()                                     # back to the buggy state first
train_out_1 = buggy_predict(serving_model, probe)
train_out_2 = buggy_predict(serving_model, probe)
assert not torch.equal(train_out_1, train_out_2), \
    "sanity: in train mode two dropout forwards should differ (32 coin flips)"

result_1 = correct_predict(serving_model, probe)
result_2 = correct_predict(serving_model, probe)
assert result_1 is not None, "fill in the stub above first"
assert torch.equal(result_1, result_2), "correct_predict must be deterministic — did you call eval()?"
assert result_1.grad_fn is None and not result_1.requires_grad, \
    "correct_predict built a graph — wrap the forward in no_grad()"

# in eval mode, dropout is identity: output must equal the bare Linear
expected = serving_model[0](probe)
assert torch.allclose(result_1, expected, atol=1e-6), "eval-mode output should be exactly the Linear's output"
print("deterministic, graph-free, dropout-disabled inference ✓")
print("Exercise 5 passed ✓")

deterministic, graph-free, dropout-disabled inference ✓
Exercise 5 passed ✓


## Exercise 6 — The Double-Softmax Bug

`BuggyClassifier`'s forward ends with `softmax` — and then training code (correctly) calls `F.cross_entropy`, which applies softmax **again**. Nothing crashes; the loss is just wrong and the gradients are squashed. Write `FixedClassifier` returning **raw logits**, and watch both the loss values and the gradient norms diverge.

**Decision you're practicing:** models return logits; softmax belongs to inference (if anywhere). The #1 loss-function gotcha.

In [12]:
class BuggyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Linear(6, 3)

    def forward(self, x):
        return torch.softmax(self.net(x), dim=-1)     # ← probabilities out of forward: the bug

class FixedClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Linear(6, 3)

    def forward(self, x):
        return self.net(x)                            # raw logits — cross_entropy softmaxes internally

torch.manual_seed(5)
buggy = BuggyClassifier()
fixed = FixedClassifier()
fixed.net.load_state_dict(buggy.net.state_dict())     # identical weights: only the forward differs

features = torch.randn(16, 6) * 3                     # scaled up so the logits have real spread
labels = torch.randint(0, 3, (16,))

loss_buggy = F.cross_entropy(buggy(features), labels)   # softmax-of-softmax
loss_fixed = F.cross_entropy(fixed(features), labels)   # softmax once, inside the loss
print(f"loss with double softmax : {loss_buggy.item():.4f}")
print(f"loss with raw logits     : {loss_fixed.item():.4f}")

loss_buggy.backward()
loss_fixed.backward()
buggy_grad_norm = buggy.net.weight.grad.norm().item()
fixed_grad_norm = fixed.net.weight.grad.norm().item()
print(f"grad norm, double softmax: {buggy_grad_norm:.5f}   ← squashed training signal")
print(f"grad norm, raw logits    : {fixed_grad_norm:.5f}")

loss with double softmax : 1.1355
loss with raw logits     : 1.5930
grad norm, double softmax: 0.36169   ← squashed training signal
grad norm, raw logits    : 2.13791


**Verification**

In [13]:
# --- Verification: Exercise 6 ---
torch.manual_seed(5)
check_buggy = BuggyClassifier()
check_fixed = FixedClassifier()
assert check_fixed is not None and hasattr(check_fixed, "net"), "implement FixedClassifier with a .net Linear"
check_fixed.net.load_state_dict(check_buggy.net.state_dict())

features = torch.randn(16, 6) * 3
labels = torch.randint(0, 3, (16,))

reference = F.cross_entropy(check_buggy.net(features), labels)      # the correct loss, from raw logits
loss_fixed = F.cross_entropy(check_fixed(features), labels)
loss_buggy = F.cross_entropy(check_buggy(features), labels)

assert torch.allclose(loss_fixed, reference, atol=1e-6), "FixedClassifier must return RAW logits"
assert not torch.allclose(loss_buggy, reference, atol=1e-3), "sanity: double softmax gives a different loss"

# gradient comparison — one clean backward per model (ch03: grads accumulate!)
for param in check_buggy.parameters():
    param.grad = None
F.cross_entropy(check_buggy(features), labels).backward()
buggy_norm = check_buggy.net.weight.grad.norm().item()

for param in check_fixed.parameters():
    param.grad = None
F.cross_entropy(check_fixed(features), labels).backward()
fixed_norm = check_fixed.net.weight.grad.norm().item()

assert buggy_norm < fixed_norm, \
    f"double softmax should squash gradients ({buggy_norm:.5f} vs {fixed_norm:.5f})"
print(f"double softmax shrank the gradient norm {fixed_norm / buggy_norm:.1f}× — silent, slow training ✓")
print("Exercise 6 passed ✓")

double softmax shrank the gradient norm 5.9× — silent, slow training ✓
Exercise 6 passed ✓


## Exercise 7 — The Loss Chooser

Three datasets, three different problems. Compute each loss with the right function and record your choice in `chooser` (allowed values: `"cross_entropy"`, `"nll_loss"`, `"bce_with_logits"`):

1. multi-class, exactly one label per sample, model gives **raw logits**
2. a pipeline that already produced **log-probabilities**
3. **multi-label** tagging — each sample can carry several tags at once

The verification also proves the family identity: `cross_entropy(z) == nll_loss(log_softmax(z))`.

**Decision you're practicing:** the loss decision table — softmax-competition vs independent sigmoids, logits vs log-probs.

In [14]:
torch.manual_seed(6)
single_label_logits  = torch.randn(8, 5)                      # (batch, classes) — raw scores
single_label_targets = torch.randint(0, 5, (8,))

ready_log_probs      = F.log_softmax(torch.randn(8, 5), dim=-1)   # already log-probabilities
log_prob_targets     = torch.randint(0, 5, (8,))

multi_label_logits   = torch.randn(8, 4)                      # (batch, tags) — raw scores
multi_label_targets  = torch.randint(0, 2, (8, 4)).float()    # several 1s per row allowed

loss_single = F.cross_entropy(single_label_logits, single_label_targets)         # logits + one true class
loss_logprob = F.nll_loss(ready_log_probs, log_prob_targets)                     # CE minus the softmax half
loss_multi = F.binary_cross_entropy_with_logits(multi_label_logits, multi_label_targets)  # independent sigmoids

chooser = {
    "single_label":   "cross_entropy",
    "have_log_probs": "nll_loss",
    "multi_label":    "bce_with_logits",
}
print(f"single-label CE       : {loss_single.item():.4f}")
print(f"NLL on log-probs      : {loss_logprob.item():.4f}")
print(f"multi-label BCE-logits: {loss_multi.item():.4f}")

# the family identity: CE really is NLL ∘ log_softmax
identity_check = F.nll_loss(F.log_softmax(single_label_logits, dim=-1), single_label_targets)
print(f"nll(log_softmax(z))   : {identity_check.item():.4f}   == CE above ✓")

single-label CE       : 1.4761
NLL on log-probs      : 2.0053
multi-label BCE-logits: 0.8779
nll(log_softmax(z))   : 1.4761   == CE above ✓


**Verification**

In [15]:
# --- Verification: Exercise 7 ---
assert loss_single is not None and loss_logprob is not None and loss_multi is not None, "fill in the stubs"

assert torch.allclose(loss_single, F.cross_entropy(single_label_logits, single_label_targets), atol=1e-6)
assert torch.allclose(loss_logprob, F.nll_loss(ready_log_probs, log_prob_targets), atol=1e-6)
assert torch.allclose(loss_multi,
                      F.binary_cross_entropy_with_logits(multi_label_logits, multi_label_targets), atol=1e-6)

# the family identity, asserted
identity = F.nll_loss(F.log_softmax(single_label_logits, dim=-1), single_label_targets)
assert torch.allclose(loss_single, identity, atol=1e-6), "CE must equal NLL(log_softmax(logits))"

expected_chooser = {"single_label": "cross_entropy", "have_log_probs": "nll_loss", "multi_label": "bce_with_logits"}
for task, expected_tool in expected_chooser.items():
    assert chooser[task] == expected_tool, f"{task}: expected {expected_tool!r}, got {chooser[task]!r}"
    print(f"✓ {task:16s} → {expected_tool}")
print("Exercise 7 passed ✓")

✓ single_label     → cross_entropy
✓ have_log_probs   → nll_loss
✓ multi_label      → bce_with_logits
Exercise 7 passed ✓


## Exercise 8 — Padding-Aware Sequence Loss

Language-model training: logits `(batch, seq, vocab)`, targets `(batch, seq)` with padding marked `-100`. Implement `sequence_cross_entropy` using the **flatten spelling** from notes §7 — and the verification proves it equals both the channels-second spelling and a hand-written loop over the real positions.

**Decision you're practicing:** CE's class-dimension convention (`(N, C)` after flattening) + `ignore_index` — the pattern under every LLM training loop.

In [16]:
def sequence_cross_entropy(logits, targets, ignore_index=-100):
    """Mean CE over non-padded positions.

    Args:
        logits: (batch, seq, vocab) raw scores
        targets: (batch, seq) int64, padded positions = ignore_index
    Returns:
        scalar loss tensor.
    Math note: mean over kept (b, t) of -log softmax(logits[b, t])[targets[b, t]]
    """
    batch_size, seq_len, vocab_size = logits.shape
    flat_logits = logits.view(-1, vocab_size)      # (B·T, V): every position becomes a "sample"
    flat_targets = targets.view(-1)                # (B·T,)
    print(f"flattened {tuple(logits.shape)} → {tuple(flat_logits.shape)}, "
          f"ignoring {(flat_targets == ignore_index).sum().item()} padded positions")
    return F.cross_entropy(flat_logits, flat_targets, ignore_index=ignore_index)

**Verification**

In [17]:
# --- Verification: Exercise 8 ---
torch.manual_seed(7)
BATCH, SEQ, VOCAB = 2, 4, 6
test_logits = torch.randn(BATCH, SEQ, VOCAB)
test_targets = torch.tensor([[3, 1, 4, -100],
                             [2, -100, -100, -100]])      # 4 real positions, 4 padded

mine = sequence_cross_entropy(test_logits, test_targets)
assert mine is not None, "fill in the stub above first"

# spelling 2: channels-second — must give the same number
channels_second = F.cross_entropy(test_logits.permute(0, 2, 1), test_targets, ignore_index=-100)
assert torch.allclose(mine, channels_second, atol=1e-6), "the two CE spellings should agree exactly"

# the definition, as a plain loop over real positions
log_probs = F.log_softmax(test_logits, dim=-1)
picked = []
for row in range(BATCH):
    for position in range(SEQ):
        if test_targets[row, position].item() != -100:
            picked.append(-log_probs[row, position, test_targets[row, position]].item())
manual = sum(picked) / len(picked)
assert abs(mine.item() - manual) < 1e-6, f"loop reference {manual:.6f} vs yours {mine.item():.6f}"
print(f"flatten spelling = channels-second spelling = hand loop = {mine.item():.6f} over {len(picked)} real tokens ✓")
print("Exercise 8 passed ✓")

flattened (2, 4, 6) → (8, 6), ignoring 4 padded positions
flatten spelling = channels-second spelling = hand loop = 3.256194 over 4 real tokens ✓
Exercise 8 passed ✓


---
## Done!

Compare your work against `solved/ch05-nlp-layer-and-loss-toolbox-solved.ipynb`.

The **model block** (ch04–ch05) is complete: containers, registration, and every layer/loss an NLP model is made of. Next is the **training block** — **ch06 — Text Data Pipeline**: `Dataset`, `DataLoader`, and the collate function that turns ragged text into exactly the padded batches exercise 8's loss consumes.